# 예제 02. 파라미터 다루기와 응답 변동성 측정

전산실험 2 · AI Agent 만들기 · **1주차** · 예상 소요 45분

## 목표
- `temperature`, `top_p`, `max_tokens`, `seed` 의 역할을 안다
- 응답을 한 글자씩 흘려보내는 **스트리밍** 을 구현한다
- 같은 프롬프트를 **30회 반복 호출**해 응답의 변동을 측정하고 시각화한다
- LLM이 확률적 시스템이라는 사실을 데이터로 확인한다

> 이 노트북의 후반부(반복 실험)는 호출이 90회 일어나 몇 분 걸립니다.
> 셀을 실행해 두고 설명을 들으세요.

In [ ]:
!pip install -q openai

import os
from openai import OpenAI

def get_secret(name, secret=True):
    # Colab Secrets -> 환경변수 -> 직접 입력. 값은 코드에 적지 않는다.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
MODEL    = "gemma4"

client = OpenAI(api_key=get_secret("KMU_API_KEY"), base_url=BASE_URL, timeout=120)

def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    return r.choices[0].message.content.strip()

print("연결 준비 완료:", MODEL)

---
## 1. `temperature` — 무작위성 손잡이

모델은 다음에 올 토큰의 **확률분포**를 계산한 뒤 그 중 하나를 고릅니다.
`temperature` 는 이 분포를 얼마나 평평하게 만들지를 정합니다.

```
temperature 낮음(0.0)  →  분포가 뾰족해짐  →  항상 최빈 토큰 선택  →  안정적, 지루함
temperature 높음(1.5)  →  분포가 평평해짐  →  드문 토큰도 선택     →  다양함, 위험함
```

에이전트는 대부분 **낮은 온도**를 씁니다.
우리가 원하는 것은 창의적인 문장이 아니라 지시한 형식을 지키는 출력이기 때문입니다.

In [ ]:
질문 = "통계학과 대학원생을 위한 스터디 그룹 이름을 하나만 지어 주세요. 이름만 답하세요."

for t in [0.0, 0.7, 1.5]:
    print(f"--- temperature = {t} ---")
    for _ in range(3):
        print("   ", ask(질문, temperature=t, max_tokens=50))
    print()

### `top_p` 는 무엇인가

`top_p=0.9` 는 "확률 상위 90%에 드는 후보들 중에서만 고른다"는 뜻입니다.
temperature와 목적이 비슷한 다른 방식의 조절 장치입니다.

> **둘을 동시에 조절하지 마세요.** 어느 쪽 효과인지 해석할 수 없게 됩니다.
> 관례적으로 `temperature` 하나만 씁니다.

In [ ]:
# top_p 를 좁히면 안전한 단어만 나옵니다
for p in [0.1, 1.0]:
    print(f"--- top_p = {p} (temperature 고정 1.0) ---")
    for _ in range(3):
        print("   ", ask(질문, temperature=1.0, top_p=p, max_tokens=50))
    print()

---
## 2. `max_tokens` — 안전장치이지 길이 조절기가 아니다

`max_tokens` 를 작게 잡으면 문장이 **중간에서 잘립니다**.
길이를 줄이고 싶으면 프롬프트로 지시해야 합니다.

In [ ]:
질문2 = "분산분석(ANOVA)이 무엇인지 설명해 주세요."

print("[max_tokens=40 으로 자름]")
print(ask(질문2, max_tokens=40))
print()
print("[프롬프트로 길이를 지시]")
print(ask(질문2 + " 정확히 두 문장으로만 답하세요.", max_tokens=300))

---
## 3. 스트리밍 — 기다리지 않고 받아 보기

응답 전체가 완성될 때까지 기다리면 답답합니다.
`stream=True` 로 요청하면 토큰이 생성되는 대로 조각조각 옵니다.

에이전트에서는 사용자에게 "지금 무언가 하고 있다"는 신호를 주는 데 씁니다.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "회귀분석의 잔차가 무엇인지 설명해 주세요."}],
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

---
## 4. 반복 호출을 안전하게 만들기

다음 실험은 90번 호출합니다. 그 중 한 번이라도 네트워크 오류가 나면
전체가 멈춰 버립니다. **재시도 함수**를 먼저 만들어 둡니다.

이런 자잘한 유틸리티가 쌓이는 것이, 다음 주에 코드를 파일로 정리해야 하는 이유입니다.

In [ ]:
import time

def ask_retry(prompt, n_retry=3, wait=2.0, **kw):
    last = None
    for attempt in range(1, n_retry + 1):
        try:
            return ask(prompt, **kw)
        except Exception as exc:
            last = exc
            if attempt < n_retry:
                print(f"  [재시도 {attempt}] {type(exc).__name__}")
                time.sleep(wait * attempt)
    raise RuntimeError(f"{n_retry}회 모두 실패") from last

print(ask_retry("1+1은?", max_tokens=20))

---
## 5. 응답 변동성 측정 — 이 수업이 통계학과 강의인 이유

> **"온도를 올리면 다양해진다"** 는 설명을 듣고 넘어가지 말고 **측정**합시다.

같은 프롬프트를 세 가지 온도에서 각각 30회 호출하고 다음을 집계합니다.

- 응답 길이(글자 수)의 평균과 표준편차
- **서로 다른 응답의 개수** (30개 중 몇 종류인가)
- 특정 키워드가 등장한 비율

몇 분 걸립니다. 실행해 두고 다음 설명을 들으세요.

In [ ]:
import pandas as pd

프롬프트 = "1부터 10 사이의 정수를 하나만 고르세요. 숫자만 답하세요."
온도들   = [0.0, 0.7, 1.2]
N        = 30

기록 = []
for t in 온도들:
    print(f"temperature={t} 수집 중", end="")
    for i in range(N):
        답 = ask_retry(프롬프트, temperature=t, max_tokens=10)
        기록.append({"temperature": t, "회차": i + 1, "응답": 답})
        if (i + 1) % 10 == 0:
            print(".", end="", flush=True)
    print(" 완료")

df = pd.DataFrame(기록)
print()
print(df.head())
print("총", len(df), "건")

In [ ]:
# 집계
df["길이"] = df["응답"].str.len()

요약 = df.groupby("temperature").agg(
    응답길이_평균=("길이", "mean"),
    응답길이_표준편차=("길이", "std"),
    고유응답수=("응답", "nunique"),
    최빈응답=("응답", lambda s: s.mode().iloc[0]),
    최빈응답_비율=("응답", lambda s: (s == s.mode().iloc[0]).mean()),
)
요약.round(3)

In [ ]:
# 온도별 응답 분포
pd.crosstab(df["응답"], df["temperature"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(온도들), figsize=(14, 4), sharey=True)

for ax, t in zip(axes, 온도들):
    부분 = df[df["temperature"] == t]["응답"].value_counts().sort_index()
    ax.bar(부분.index.astype(str), 부분.values)
    ax.set_title(f"temperature = {t}\n고유 응답 {부분.size}종")
    ax.set_xlabel("response")
    ax.tick_params(axis="x", rotation=45)

axes[0].set_ylabel("count")
plt.suptitle("Same prompt, 30 calls each")
plt.tight_layout()
plt.show()

### 무엇이 보이는가

두 가지를 확인하세요.

**1. `temperature=0` 인데도 응답이 완전히 같지 않을 수 있습니다.**
서버 구현, 배치 처리, 부동소수점 연산 순서 등의 이유로 미세한 비결정성이 남습니다.
`seed` 파라미터를 지원하는 서버라면 조금 나아지지만 완전한 재현은 보장되지 않습니다.

**2. 온도가 올라갈수록 고유 응답 수가 늘고 최빈응답 비율이 떨어집니다.**

> ### 이것이 뜻하는 바
> 우리가 만들 에이전트는 **결정론적 프로그램이 아니라 확률적 시스템**입니다.
>
> - 한 번 돌려서 잘 됐다고 "동작한다"고 말할 수 없습니다
> - 성능은 `정확도 80%` 가 아니라 `80%, 95% CI [68%, 89%]` 로 보고해야 합니다
> - 두 프롬프트의 우열을 가리려면 **검정**이 필요합니다
>
> 이 문제를 **11주차(에이전트 평가)** 에서 본격적으로 다룹니다.

In [ ]:
# seed 를 지원하는 서버인지 확인 (지원하지 않으면 오류가 나거나 무시됩니다)
try:
    답들 = [ask(프롬프트, temperature=1.0, seed=42, max_tokens=10) for _ in range(5)]
    print("seed=42 로 5회:", 답들)
    print("모두 동일한가?", len(set(답들)) == 1)
except Exception as exc:
    print("이 서버는 seed 를 지원하지 않습니다:", type(exc).__name__)

---
## 연습문제

**문제 1.** 위 실험을 `temperature=0.0` 과 `temperature=1.2` 두 조건으로만,
프롬프트를 "1부터 100 사이"로 바꾸어 각 50회씩 다시 수행하시오.
고유 응답 수의 차이가 더 뚜렷해지는가?

**문제 2.** 모델이 고른 숫자의 분포는 균등한가?
`temperature=1.2` 조건의 응답에 대해 균등분포 적합도 카이제곱 검정을 수행하시오.
(`scipy.stats.chisquare`)

**문제 3.** 응답 길이의 표준편차가 온도에 따라 커지는지 검정하시오.
어떤 검정을 쓰는 것이 적절한가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
범위가 넓어지면 온도 0에서도 후보가 늘어날 수 있지만, 여전히 특정 숫자(7, 42 등)에
강하게 쏠리는 경향이 관찰됩니다. 모델은 균등추출기가 아니라 **학습 데이터의 빈도를
반영하는 생성기**이기 때문입니다.

**문제 2.**
```python
from scipy.stats import chisquare
관측 = df[df["temperature"] == 1.2]["응답"].value_counts().sort_index()
print(chisquare(관측.values))
```
대개 균등분포가 강하게 기각됩니다. 7이나 3 같은 숫자가 과대표집됩니다.
**"LLM에게 난수를 뽑게 하지 마라"** 는 실무 규칙의 근거입니다.
난수가 필요하면 파이썬으로 뽑아서 넘겨 주십시오. (5주차 도구 단원과 연결)

**문제 3.**
등분산 검정입니다. 정규성을 가정할 수 있으면 Bartlett, 아니면 Levene 검정을 씁니다.
```python
from scipy.stats import levene
그룹들 = [df[df["temperature"] == t]["길이"].values for t in 온도들]
print(levene(*그룹들))
```
표본이 30이므로 정규성 가정이 불안하다면 Levene 쪽이 안전합니다.

</details>

---
## 정리

- `temperature` 는 무작위성 손잡이. **에이전트는 낮게** 쓴다
- `max_tokens` 는 안전장치일 뿐, 길이는 프롬프트로 지시한다
- 반복 호출에는 **재시도**를 반드시 넣는다
- LLM은 확률적 시스템이다. 한 번 성공은 성공이 아니다

**다음 노트북** → [`03_provider_switch.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week01/03_provider_switch.ipynb)